## Patient IDs: HUPA0024P, HUPA0025P, HUPA0026P, HUPA0027P, HUPA0028

In [11]:
import pandas as pd
import glob
import os

# Get 5 CSV files from the folder
cgm_5files_tm = glob.glob("/Users/tejaswinimode/Desktop/NUMPY/PyHackathon/CGM_5files/*.csv")

# Add Patient_ID column & merge all 5 files 
dfs = []
for file in cgm_5files_tm :
    if os.path.isfile(file):
        df = pd.read_csv(file, sep=';')
        df['patient_id'] = os.path.basename(file).replace('.csv', '')
        dfs.append(df)

df_CGM5_TM = pd.concat(dfs, ignore_index=True)

# Move patient_id to first column
cols = ['patient_id'] + [col for col in df_CGM5_TM.columns if col != 'patient_id']
df_CGM5_TM = df_CGM5_TM[cols]

df_CGM5_TM

,patient_id,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,HUPA0025P,2020-01-16T13:45:00,108.000000,17.58320,80.828571,54.0,0.000,0.0,0.0
1,HUPA0025P,2020-01-16T13:50:00,103.666667,11.34400,74.729730,22.0,0.000,0.0,0.0
2,HUPA0025P,2020-01-16T13:55:00,99.333333,13.18740,76.444444,14.0,0.000,0.0,0.0
3,HUPA0025P,2020-01-16T14:00:00,95.000000,12.76200,80.151515,17.0,0.000,0.0,0.0
4,HUPA0025P,2020-01-16T14:05:00,98.333333,15.59800,74.952381,71.0,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...
238716,HUPA0026P,2020-10-10T23:35:00,156.666667,4.55436,79.600000,0.0,0.052,0.0,0.0
238717,HUPA0026P,2020-10-10T23:40:00,158.333333,4.55436,78.137931,0.0,0.052,0.0,0.0
238718,HUPA0026P,2020-10-10T23:45:00,160.000000,4.21700,75.657143,0.0,0.052,0.0,0.0
238719,HUPA0026P,2020-10-10T23:50:00,159.666667,4.38568,75.096774,0.0,0.052,0.0,0.0


In [12]:
# Convert all column names to title case
df_CGM5_TM.columns = df_CGM5_TM.columns.str.title()

# Check Existing datatype of columns
df_CGM5_TM.dtypes

# Convert 'time' column from object to datetime format for time-based operation
df_CGM5_TM['Time'] = pd.to_datetime(df_CGM5_TM['Time'])

# Convert 'steps' column from float64 to int as steps are always whole numbers
df_CGM5_TM['Steps'] = df_CGM5_TM['Steps'].astype(int)

df_CGM5_TM.dtypes



Patient_Id                        object
Time                      datetime64[ns]
Glucose                          float64
Calories                         float64
Heart_Rate                       float64
Steps                              int64
Basal_Rate                       float64
Bolus_Volume_Delivered           float64
Carb_Input                       float64
dtype: object

In [13]:
# Check nulls
df_CGM5_TM.isnull().sum() # There are no null values

Patient_Id                0
Time                      0
Glucose                   0
Calories                  0
Heart_Rate                0
Steps                     0
Basal_Rate                0
Bolus_Volume_Delivered    0
Carb_Input                0
dtype: int64

In [14]:
# Round all float64 columns to 2 decimal places
float_cols = df_CGM5_TM.select_dtypes(include='float64').columns
df_CGM5_TM[float_cols] = df_CGM5_TM[float_cols].round(2)


df_CGM5_TM.tail()

,Patient_Id,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input
238716,HUPA0026P,2020-10-10 23:35:00,156.67,4.55,79.60,0,0.05,0.0,0.0
238717,HUPA0026P,2020-10-10 23:40:00,158.33,4.55,78.14,0,0.05,0.0,0.0
238718,HUPA0026P,2020-10-10 23:45:00,160.00,4.22,75.66,0,0.05,0.0,0.0
238719,HUPA0026P,2020-10-10 23:50:00,159.67,4.39,75.10,0,0.05,0.0,0.0
238720,HUPA0026P,2020-10-10 23:55:00,159.33,4.39,74.63,0,0.05,0.0,0.0


In [15]:
# Check total number of duplicate rows
print("Total duplicates:", df_CGM5_TM.duplicated().sum())

Total duplicates: 0


In [16]:
# No column should have negative values
numeric_cols = df_CGM5_TM.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    neg = (df_CGM5_TM[col] < 0).sum()
    print(f"{col}: {neg} negative values")

Glucose: 0 negative values
Calories: 0 negative values
Heart_Rate: 0 negative values
Steps: 0 negative values
Basal_Rate: 0 negative values
Bolus_Volume_Delivered: 0 negative values
Carb_Input: 0 negative values


In [17]:
# Sort data by patient and time for proper time series analysis
df_CGM5_TM = df_CGM5_TM.sort_values(['Patient_Id', 'Time']).reset_index(drop=True)
df_CGM5_TM.head()

,Patient_Id,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input
0,HUPA0024P,2020-01-20 11:30:00,87.00,8.19,67.36,20,0.07,0.0,0.0
1,HUPA0024P,2020-01-20 11:35:00,86.00,17.99,75.25,107,0.07,0.0,0.0
2,HUPA0024P,2020-01-20 11:40:00,85.00,17.41,76.78,123,0.07,0.0,0.0
3,HUPA0024P,2020-01-20 11:45:00,84.00,10.38,83.67,30,0.07,0.0,0.0
4,HUPA0024P,2020-01-20 11:50:00,85.33,14.53,80.29,74,0.07,0.0,0.0


## Save Data into .csv File

In [18]:
df_CGM5_TM.to_csv(
    r"/Users/tejaswinimode/Desktop/NUMPY/PyHackathon/GitHub/Team2_-PyQueens_-Python-Hackathon-_MAY-2026/Tejaswini/CleanedData_TM.csv",
    index=False
)

## Data Cleaning for patient_sleep_demographics files

In [19]:
import pandas as pd

df_sl = pd.read_csv("/Users/tejaswinimode/Desktop/NUMPY/PyHackathon/CGM_Data/HUPA-UC Diabetes Dataset/T1DM_patient_sleep_demographics_with_race.csv", sep=',',engine = 'python')
df_sl



,Patient_ID,Age,Gender,Race,Average Sleep Duration (hrs),Sleep Quality (1-10),% with Sleep Disturbances
0,HUPA0001P,34,Male,Other,6.3,4.5,80
1,HUPA0002P,49,Male,Hispanic,6.6,4.4,40
2,HUPA0003P,64,Male,Black,5.3,5.2,70
3,HUPA0004P,34,Female,Native American,5.2,6.9,60
4,HUPA0005P,49,Male,Native American,5.8,7.9,30
5,HUPA0006P,35,Male,White,6.6,4.2,60
6,HUPA0007P,67,Male,Native American,7.1,6.0,80
7,HUPA0009P,65,Female,Other,6.6,4.6,40
8,HUPA0010P,22,Male,Asian,7.1,5.5,50
9,HUPA0011P,63,Female,Other,5.6,4.7,60


In [20]:
# Check Existing datatype of columns
df_sl.dtypes   # data looks clean

Patient_ID                       object
Age                               int64
Gender                           object
Race                             object
Average Sleep Duration (hrs)    float64
Sleep Quality (1-10)            float64
% with Sleep Disturbances         int64
dtype: object